# PatchTST + 매장별 적응 (Per-series K-fold Validation)
- Train: `./dataset/train.csv`  |  Test: `./dataset/TEST_*.csv`
- Out: `./artifacts/best.ckpt`, `./artifacts/adapters/<store>/adapter.pt`, `./result/submission_patchtst_storeadapt.csv`
- 검증: **store_menu별 시간 K-fold + embargo** 합집합 방식

In [62]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [63]:
# === 1) 설치 ===
import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for p in ["numpy","pandas","scikit-learn","tqdm","pyyaml","torch","matplotlib"]:
    pip_install(p)
print("설치 완료")

설치 완료


In [64]:
# === 2) Import / Config ===
import os, json, math, random, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

# 경로
DATA_DIR = Path("./dataset")
RESULT_DIR = Path("./result")
ARTIFACTS = Path("./artifacts_patchTST_gpt")
ADAPTER_DIR = ARTIFACTS / "adapters"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    # PatchTST
    L: int = 28
    H: int = 7
    patch_len: int = 7
    stride: int = 3
    d_model: int = 256
    nhead: int = 8
    num_layers: int = 4
    ff_dim: int = 512
    dropout: float = 0.1
    # 학습
    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 1e-4
    max_epochs: int = 100
    early_stop_patience: int = 8
    # 검증(K-fold)
    n_splits: int = 5
    embargo_days: int = 35    # ≈ L + H 이상 권장
    # 2단계 적응
    adapt_epochs: int = 10
    adapt_lr: float = 5e-4
    adapt_decay_alpha: float = 5.0
    # 옵션 
    use_film: bool = True
    use_bias_scale: bool = True
    use_lora: bool = False
    # 산출물
    save_best_ckpt: str = str(ARTIFACTS / "best.ckpt")
    log_csv: str = "training_log.csv"
    config_json: str = "config_gpt_storeadapt.json"
    # loss function
    loss_type: str = "mae"          # ["mae","rmse","huber","smape","mae_log1p"]
    huber_delta: float = 1.0
    smape_eps: float = 1e-3
CFG = Config()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)
print(CFG)

Config(seed=42, device='cuda', L=28, H=7, patch_len=7, stride=3, d_model=256, nhead=8, num_layers=4, ff_dim=512, dropout=0.1, batch_size=128, lr=0.0003, weight_decay=0.0001, max_epochs=100, early_stop_patience=8, n_splits=5, embargo_days=35, adapt_epochs=10, adapt_lr=0.0005, adapt_decay_alpha=5.0, use_film=True, use_bias_scale=True, use_lora=False, save_best_ckpt='artifacts_patchTST_gpt/best.ckpt', log_csv='training_log.csv', config_json='config_gpt_storeadapt.json', loss_type='mae', huber_delta=1.0, smape_eps=0.001)


In [ ]:
# === 3) Utils: metrics / features / windows / scaler ===
EPS = 1e-6

def loss_per_sample(y: torch.Tensor, yhat: torch.Tensor, cfg) -> torch.Tensor:
    # 반환: (B,)  — 각 샘플의 평균 시계열 손실
    if cfg.loss_type == "mae":
        return torch.mean(torch.abs(yhat - y), dim=1)
    if cfg.loss_type == "rmse":
        return torch.sqrt(torch.mean((yhat - y)**2, dim=1) + 1e-8)
    if cfg.loss_type == "huber":
        d = torch.abs(yhat - y)
        delta = cfg.huber_delta
        per_t = torch.where(d <= delta, 0.5*(d**2)/delta, d - 0.5*delta)
        return torch.mean(per_t, dim=1)
    if cfg.loss_type == "mae_log1p":
        # 비음수 가정. 음수 예측은 0으로 클립 후 log1p
        yh = torch.log1p(torch.clamp(yhat, min=0))
        yt = torch.log1p(torch.clamp(y,    min=0))
        return torch.mean(torch.abs(yh - yt), dim=1)
    # 안전한 sMAPE(학습용)
    den = torch.clamp(torch.abs(yhat) + torch.abs(y), min=cfg.smape_eps)
    return torch.mean(2.0*torch.abs(yhat - y)/den, dim=1)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_pred - y_true)
    den = np.abs(y_pred) + np.abs(y_true) + EPS
    return 100.0 * np.mean(2.0 * num / den)

def smape_torch(y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
    num = torch.abs(y_pred - y_true)
    den = torch.abs(y_pred) + torch.abs(y_true) + 1e-6
    return 100.0 * torch.mean(2.0 * num / den)

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    dow = df["date"].dt.dayofweek
    df["is_weekend"] = (dow >= 5).astype(int)
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    mon = df["date"].dt.month
    df["mon_sin"] = np.sin(2 * np.pi * mon / 12.0)
    df["mon_cos"] = np.cos(2 * np.pi * mon / 12.0)
    df["is_eom"] = df["date"].dt.is_month_end.astype(int)
    df["is_payday25"] = (df["date"].dt.day == 25).astype(int)
    if "store" in df.columns and "sales" in df.columns:
        df["_store_daily_total"] = df.groupby(["store","date"])["sales"].transform("sum")
        df["_store_ma7"] = df.groupby("store")["_store_daily_total"].transform(lambda s: s.rolling(7, min_periods=1).mean())
        df["_store_ma14"] = df.groupby("store")["_store_daily_total"].transform(lambda s: s.rolling(14, min_periods=1).mean())
        df["_store_ratio"] = (df["sales"] / (df["_store_daily_total"] + EPS)).fillna(0.0)
    else:
        df["_store_daily_total"] = 0.0
        df["_store_ma7"] = 0.0
        df["_store_ma14"] = 0.0
        df["_store_ratio"] = 0.0
    return df

def clean_and_unify(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "store_menu" not in df.columns:
        if "store_menu_id" in df.columns:
            df["store_menu"] = df["store_menu_id"]
        else:
            df["store_menu"] = df["store"].astype(str) + "_" + df["menu"].astype(str)
    df["date"] = pd.to_datetime(df["date"])
    if "sales" in df.columns:
        df["sales"] = df["sales"].clip(lower=0)
    df = df.sort_values(["store_menu", "date"]).reset_index(drop=True)
    return df

FEATURE_COLS = [
    "is_weekend","dow_sin","dow_cos","mon_sin","mon_cos","is_eom","is_payday25",
    "_store_daily_total","_store_ma7","_store_ma14","_store_ratio",
]

def build_series_windows(df: pd.DataFrame, L: int, H: int):
    Xs, Ys, metas, ends = [], [], [], []
    for sm, g in df.groupby("store_menu"):
        g = g.sort_values("date").reset_index(drop=True)
        for c in FEATURE_COLS:
            if c not in g.columns: g[c] = 0.0
        feats = ["sales"] + FEATURE_COLS
        arr = g[feats].astype(float).values
        dates = g["date"].values
        n = len(g)
        for i in range(0, n - (L + H) + 1):
            x = arr[i:i+L]
            y = g["sales"].values[i+L:i+L+H].astype(float)
            Xs.append(x)
            Ys.append(y)
            metas.append({"store": g.loc[i, "store"] if "store" in g.columns else "NA",
                          "store_menu": sm})
            ends.append(pd.to_datetime(dates[i+L-1]))
    return Xs, Ys, metas, ends

def build_samples_meta(meta_list, end_dates: List[pd.Timestamp], H: int) -> pd.DataFrame:
    s = []
    for i, m in enumerate(meta_list):
        start = end_dates[i] + pd.Timedelta(days=1)
        end = end_dates[i] + pd.Timedelta(days=H)
        s.append((i, m["store"], m["store_menu"], start, end))
    return pd.DataFrame(s, columns=["idx","store","store_menu","target_start_date","target_end_date"])

class GlobalStandardScaler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.fitted = False
        self.feat_names = ["sales"] + FEATURE_COLS

    def fit(self, X_list: List[np.ndarray]):
        X = np.concatenate(X_list, axis=0)
        self.scaler.fit(X)
        self.fitted = True

    def transform(self, X: np.ndarray) -> np.ndarray:
        return self.scaler.transform(X)

    def save(self, path: Path):
        np.savez(path, mean=self.scaler.mean_, scale=self.scaler.scale_)

    def load(self, path: Path):
        obj = np.load(path)
        self.scaler.mean_ = obj["mean"]
        self.scaler.scale_ = obj["scale"]
        self.fitted = True


In [66]:
# === 4) Dataset / Collate ===
class WindowDataset(Dataset):
    def __init__(self, X_list, Y_list, meta_list, indices, scaler: GlobalStandardScaler, fit_scaler: bool=False):
        self.meta = [meta_list[i] for i in indices]
        self.X_raw = [X_list[i].copy() for i in indices]
        self.Y = [Y_list[i].copy() for i in indices]
        if fit_scaler:
            scaler.fit([x for x in self.X_raw])
        self.scaler = scaler

    def __len__(self): return len(self.X_raw)

    def __getitem__(self, idx):
        x = self.scaler.transform(self.X_raw[idx])
        y = self.Y[idx]
        store = self.meta[idx]["store"]
        store_menu = self.meta[idx]["store_menu"]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), store, store_menu

def collate_fn(batch):
    xs, ys, stores, sms = zip(*batch)
    x = torch.stack(xs, dim=0)
    y = torch.stack(ys, dim=0)
    return x, y, list(stores), list(sms)

In [67]:
# === 5) PatchTST + Store adaptation ===
class FiLM(nn.Module):
    def __init__(self, emb_dim: int, hidden: int, target_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(emb_dim, hidden), nn.ReLU(), nn.Linear(hidden, 2*target_dim))
    def forward(self, e):
        gb = self.net(e); return gb.chunk(2, dim=-1)

class PatchTSTBackbone(nn.Module):
    def __init__(self, input_dim: int, cfg):
        super().__init__()
        self.cfg = cfg
        self.patch_proj = nn.Linear(cfg.patch_len * input_dim, cfg.d_model)
        enc = nn.TransformerEncoderLayer(d_model=cfg.d_model, nhead=cfg.nhead,
                                         dim_feedforward=cfg.ff_dim, dropout=cfg.dropout,
                                         batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=cfg.num_layers)
        self.head = nn.Linear(cfg.d_model, cfg.H)

    def forward(self, x):
        B, L, Din = x.shape; P, S = self.cfg.patch_len, self.cfg.stride
        patches = []
        for st in range(0, L - P + 1, S):
            seg = x[:, st:st+P, :].reshape(B, -1)
            patches.append(seg)
        H = torch.stack(patches, dim=1)        # (B,N,P*Din)
        N = H.size(1)
        H = self.patch_proj(H)
        pos = torch.arange(N, device=x.device).unsqueeze(0).unsqueeze(-1)
        H = H + torch.sin(pos / 10000.0).repeat(1,1,self.cfg.d_model)
        Z = self.encoder(H)
        z_last = Z[:, -1, :]                   # (B,d_model)
        out = self.head(z_last)                 # (B,H)
        return out, z_last

class StoreAdaptiveWrapper(nn.Module):
    def __init__(self, backbone: PatchTSTBackbone, store_to_idx: Dict[str,int], cfg):
        super().__init__()
        self.backbone = backbone; self.cfg = cfg
        self.store_to_idx = store_to_idx; self.num_stores = len(store_to_idx)
        emb_dim = 64
        self.store_emb = nn.Embedding(self.num_stores, emb_dim)
        self.film = FiLM(emb_dim, 64, backbone.cfg.d_model) if cfg.use_film else None
        self.out_affine = nn.Linear(emb_dim, 2) if cfg.use_bias_scale else None

    def forward(self, x, store_names: List[str]):
        out, z = self.backbone(x)
        idx = torch.tensor([self.store_to_idx.get(s, 0) for s in store_names], device=x.device)
        e = self.store_emb(idx)
        if self.film is not None:
            gamma, beta = self.film(e); out = self.backbone.head(gamma * z + beta)
        if self.out_affine is not None:
            g, b = self.out_affine(e).chunk(2, dim=-1); out = (1.0 + g) * out + b
        return out

    def adapter_state_dict(self, store_name: str):
        d = {}; si = torch.tensor(self.store_to_idx.get(store_name, 0))
        with torch.no_grad():
            d["store_emb.weight"] = self.store_emb.weight[si].detach().cpu().clone()
        return d

    def load_adapter_state(self, store_name: str, state: Dict[str, torch.Tensor]):
        si = self.store_to_idx.get(store_name, 0)
        with torch.no_grad():
            self.store_emb.weight[si].copy_(state["store_emb.weight"].to(self.store_emb.weight.device))

In [68]:
# === 6) Train / Eval ===
class EarlyStopper:
    def __init__(self, patience=8, mode="min"):
        self.patience = patience; self.best=None; self.cnt=0; self.mode=mode
    def step(self, metric):
        if self.best is None: self.best=metric; return False
        imp = metric < self.best if self.mode=="min" else metric > self.best
        if imp: self.best=metric; self.cnt=0
        else: self.cnt += 1
        return self.cnt > self.patience

def evaluate(model, loader, device):
    model.eval(); loss_sum=0.0; n=0
    with torch.no_grad():
        for x, y, stores, _ in loader:
            x=x.to(device); y=y.to(device)
            yhat = model(x, stores)
            loss = loss_per_sample(y, yhat, CFG).mean()
            loss_sum += float(loss.item()) * y.size(0); n += y.size(0)
    m = loss_sum/max(n,1)
    return m

def train_global(model, train_loader, val_loader, device, cfg, save_path: str, log_path: str):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    stopper = EarlyStopper(cfg.early_stop_patience, "min")
    rows = []; best_state=None; best=float("inf")
    for epoch in range(1, cfg.max_epochs+1):
        model.train()
        for x,y,stores,_ in tqdm(train_loader, desc=f"[Global] Epoch {epoch}", leave=False):
            x=x.to(device); y=y.to(device)
            loss = smape_torch(y, model(x, stores))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_sm = evaluate(model, val_loader, device)
        rows.append({"stage":"global","epoch":epoch,"val_smape":val_sm})
        print(f"Epoch {epoch:03d} | val sMAPE={val_sm:.3f}")
        if val_sm < best:
            best = val_sm; best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            torch.save(best_state, save_path)
        if stopper.step(val_sm): print("Early stopping"); break
    pd.DataFrame(rows).to_csv(log_path, index=False)
    if best_state is not None: model.load_state_dict(best_state)

In [69]:
# === 7) Per-series K-fold split with embargo ===
def build_time_kfold_splits_by_series(samples: pd.DataFrame, n_splits: int, L: int, H: int, embargo_days: int, verbose=True):
    s = samples.sort_values(['store_menu','target_start_date']).reset_index(drop=False).rename(columns={'index':'_orig_idx'})
    fold_tr = [set() for _ in range(n_splits)]
    fold_va = [set() for _ in range(n_splits)]

    for sm, g in s.groupby("store_menu"):
        g = g.sort_values("target_start_date").reset_index(drop=True)
        uniq = pd.Series(g["target_start_date"].unique()).sort_values().to_list()
        warmup = L + H + embargo_days
        earliest = (pd.Timestamp(uniq[0]) + pd.Timedelta(days=warmup)) if len(uniq)>0 else None
        cands = [d for d in uniq if earliest is not None and d >= earliest]

        local_splits = min(n_splits, max(1, len(cands))) if cands else 1
        bins = (np.array_split(np.array(cands), local_splits) if cands else [np.array([], dtype='datetime64[ns]')])

        for k in range(n_splits):
            if k >= len(bins): continue
            val_dates = bins[k]
            if len(val_dates)==0: continue
            val_start = pd.Timestamp(val_dates[0])
            # local masks
            val_mask = g["target_start_date"].isin(val_dates)
            val_loc = g.index[val_mask].to_numpy()
            cutoff = val_start - pd.Timedelta(days=embargo_days)
            train_mask = g["target_end_date"] < cutoff
            train_loc = g.index[train_mask].to_numpy()
            if len(train_loc)==0 or len(val_loc)==0: continue
            val_idx_global = g.loc[val_loc, "_orig_idx"].to_numpy()
            trn_idx_global = g.loc[train_loc, "_orig_idx"].to_numpy()
            fold_tr[k].update(trn_idx_global.tolist())
            fold_va[k].update(val_idx_global.tolist())

    folds = []
    for k in range(n_splits):
        tr = np.array(sorted(fold_tr[k]), dtype=int)
        va = np.array(sorted(fold_va[k]), dtype=int)
        if len(tr)>0 and len(va)>0:
            folds.append((tr,va))
            if verbose: print(f"[Fold {len(folds)}/{n_splits}] train={len(tr):,}, val={len(va):,}")
        else:
            if verbose: print(f"[Skip] Fold {k+1}: train={len(tr)}, val={len(va)}")

    if len(folds)<=1 and embargo_days>0:
        if verbose: print("[Info] Too few folds. Relax embargo and retry.")
        return build_time_kfold_splits_by_series(samples, n_splits, L, H, max(L, embargo_days//2), verbose)
    return folds

In [70]:
# === 8) 매장별 적응 ===
def compute_recency_weights(end_dates: List[pd.Timestamp], alpha: float) -> np.ndarray:
    ed = pd.to_datetime(list(end_dates))
    if len(ed) == 0:
        return np.array([], dtype=np.float32)
    ed_min = ed.min()
    span = (ed.max() - ed_min)
    if span == pd.Timedelta(0):
        span = pd.Timedelta(seconds=1)
    t = ((ed - ed_min) / span).to_numpy(dtype=float)  # ndarray
    w = np.exp(alpha * t).astype(np.float32)
    w /= (float(w.mean()) + 1e-8)  # 평균 1로 정규화
    return w



def train_store_adaptation(model, X_list, Y_list, meta_list, end_dates, device, cfg, store_name: str, scaler: GlobalStandardScaler):
    idxs = [i for i,m in enumerate(meta_list) if m["store"] == store_name]
    if len(idxs) == 0:
        return

    ds = WindowDataset(X_list, Y_list, meta_list, idxs, scaler, fit_scaler=False)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)  # shuffle=False

    for p in model.backbone.parameters():
        p.requires_grad = False
    opt = torch.optim.AdamW([p for p in model.store_emb.parameters()], lr=cfg.adapt_lr, weight_decay=0.0)

    w_all = compute_recency_weights([end_dates[i] for i in idxs], cfg.adapt_decay_alpha)

    model.train()
    for epoch in range(1, cfg.adapt_epochs+1):
        it = 0  # 매 epoch마다 리셋
        for x, y, stores, _ in tqdm(dl, desc=f"[Adapt:{store_name}] Epoch {epoch}", leave=False):
            x = x.to(device); y = y.to(device)
            yhat = model(x, stores)
            loss_vec = loss_per_sample(y, yhat, CFG)   # (B,)


            # per-sample sMAPE
            num = torch.abs(yhat - y)
            den = torch.abs(yhat) + torch.abs(y) + 1e-6
            loss_vec = 100.0 * torch.mean(2.0 * num / den, dim=1)  # (B,)

            bs = y.size(0)
            seg = w_all[it:it+bs]
            if len(seg) < bs:  # 마지막 배치 보정
                seg = np.pad(seg, (0, bs - len(seg)), mode="edge")
            seg_t = torch.tensor(seg, device=device, dtype=torch.float32)

            loss = torch.mean(loss_vec * seg_t)

            it += bs
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

    out_dir = ADAPTER_DIR / store_name
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.adapter_state_dict(store_name), out_dir / "adapter.pt")


In [71]:
# === 9) 추론 ===
def predict_for_test_files(model, device, cfg, scaler: GlobalStandardScaler, sample_sub_path: Path, out_csv: Path):
    sub = pd.read_csv(sample_sub_path)
    if sub.columns[0].lower() in {"date","ds"}:
        sub = sub.set_index(sub.columns[0])
    pred_map = {}
    test_files = sorted(DATA_DIR.glob("TEST_*.csv"))
    print("테스트 파일:", [p.name for p in test_files])
    feats = ["sales"] + FEATURE_COLS
    for fp in test_files:
        df_t = pd.read_csv(fp); df_t = clean_and_unify(df_t); df_t = add_time_features(df_t)
        for sm, g in df_t.groupby("store_menu"):
            g = g.sort_values("date")
            x = g[feats].astype(float).values
            if len(x) != cfg.L:
                pad = np.repeat(x[-1:], cfg.L - len(x), axis=0); x = np.concatenate([x, pad], axis=0)
            xt = torch.tensor(scaler.transform(x), dtype=torch.float32, device=device).unsqueeze(0)
            store = g["store"].iloc[0] if "store" in g.columns else "NA"
            yhat = model(xt, [store]).detach().cpu().numpy().reshape(-1); yhat = np.clip(yhat, 0, None)
            last_day = pd.to_datetime(g["date"].iloc[-1])
            tgt_dates = pd.date_range(last_day + pd.Timedelta(days=1), periods=cfg.H, freq="D")
            for d, v in zip(tgt_dates, yhat): pred_map.setdefault(d, {})[sm] = float(v)
    cols = [c for c in sub.columns]; all_dates = sorted(pred_map.keys())
    out = pd.DataFrame(index=all_dates, columns=cols, dtype=float)
    for d,m in pred_map.items():
        for c in cols:
            if c in m: out.loc[d, c] = m[c]
    out = out.fillna(0.0); out.index.name = sub.index.name if sub.index.name else "date"
    out.to_csv(out_csv, encoding="utf-8-sig"); print("저장:", out_csv)

In [72]:
# === 10) run_all(): K-fold 학습 → 최적 fold ckpt 선택 → 적응/추론 ===
def run_all():
    with open(CFG.config_json, "w", encoding="utf-8") as f:
        json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)

    # 1) Load & features
    df = pd.read_csv(DATA_DIR / "train.csv")
    df = clean_and_unify(df); df = add_time_features(df)

    # 2) Windows
    X_list, Y_list, meta_list, end_dates = build_series_windows(df, CFG.L, CFG.H)
    samples = build_samples_meta(meta_list, end_dates, CFG.H)

    # 3) Folds
    folds = build_time_kfold_splits_by_series(samples, CFG.n_splits, CFG.L, CFG.H, CFG.embargo_days, verbose=True)

    # 4) Store index
    stores = sorted({m["store"] for m in meta_list})
    store_to_idx = {s:i for i,s in enumerate(stores)}
    input_dim = 1 + len(FEATURE_COLS)

    # 5) K-fold train
    fold_scores = []; best_fold=None; best_score=float("inf"); best_scaler=None
    for fi,(tr_idx, va_idx) in enumerate(folds, start=1):
        print(f"\n==== Fold {fi}/{len(folds)} ====")
        scaler_f = GlobalStandardScaler()
        ds_tr = WindowDataset(X_list, Y_list, meta_list, tr_idx, scaler_f, fit_scaler=True)
        ds_va = WindowDataset(X_list, Y_list, meta_list, va_idx, scaler_f, fit_scaler=False)
        dl_tr = DataLoader(ds_tr, batch_size=CFG.batch_size, shuffle=True, collate_fn=collate_fn)
        dl_va = DataLoader(ds_va, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn)

        model = StoreAdaptiveWrapper(PatchTSTBackbone(input_dim, CFG), store_to_idx, CFG).to(CFG.device)
        ckpt_f = ARTIFACTS / f"best_fold{fi}.ckpt"
        log_f  = f"training_log_fold{fi}.csv"
        train_global(model, dl_tr, dl_va, CFG.device, CFG, save_path=str(ckpt_f), log_path=log_f)
        # 최종 fold 성능 기록
        val_sm = evaluate(model, dl_va, CFG.device)
        fold_scores.append({"fold":fi,"val_smape":float(val_sm)})
        print(f"[Fold {fi}] val sMAPE={val_sm:.3f}")
        if val_sm < best_score:
            best_score = val_sm; best_fold = fi; best_scaler = scaler_f

    # 6) Select best fold → save as global best
    if best_fold is None: raise RuntimeError("No valid folds built.")
    print(f"\nBest fold = {best_fold} | sMAPE={best_score:.3f}")
    # Save scaler of best fold
    best_scaler.save(ARTIFACTS / "scaler_best.npz")
    # Copy ckpt
    src = ARTIFACTS / f"best_fold{best_fold}.ckpt"
    shutil.copyfile(src, CFG.save_best_ckpt)
    # Load best model for adaptation/inference
    model = StoreAdaptiveWrapper(PatchTSTBackbone(input_dim, CFG), store_to_idx, CFG).to(CFG.device)
    state = torch.load(CFG.save_best_ckpt, map_location=CFG.device); model.load_state_dict(state)

    # 7) Per-store adaptation
    for s in stores:
        train_store_adaptation(model, X_list, Y_list, meta_list, end_dates, CFG.device, CFG, s, best_scaler)

    # 8) Inference
    out_csv = RESULT_DIR / "submission_patchtst_storeadapt.csv"
    predict_for_test_files(model, CFG.device, CFG, best_scaler, sample_sub_path=RESULT_DIR/"sample_submission.csv", out_csv=out_csv)

    # 9) Logs
    #pd.DataFrame(fold_scores).to_csv("fold_scores.csv", index=False)
    #print("완료.")

print("run_all() 준비됨")

run_all() 준비됨


In [73]:
# === 11) 실행 ===
run_all()
print("Ready")

[Fold 1/5] train=5,597, val=16,598
[Fold 2/5] train=22,195, val=16,598
[Fold 3/5] train=38,793, val=16,598
[Fold 4/5] train=55,391, val=16,405
[Fold 5/5] train=71,796, val=16,405

==== Fold 1/5 ====


Epoch 001 | val sMAPE=8.546


Epoch 002 | val sMAPE=10.821


Epoch 003 | val sMAPE=13.897


Epoch 004 | val sMAPE=9.671


...

Epoch 007 | val sMAPE=7.982


Epoch 008 | val sMAPE=8.737


Epoch 009 | val sMAPE=8.575


Epoch 010 | val sMAPE=8.306
Early stopping
[Fold 5] val sMAPE=7.783

Best fold = 2 | sMAPE=6.159


테스트 파일: ['TEST_00.csv', 'TEST_01.csv', 'TEST_02.csv', 'TEST_03.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_06.csv', 'TEST_07.csv', 'TEST_08.csv', 'TEST_09.csv']
저장: result/submission_patchtst_storeadapt.csv
Ready


In [ ]:
import pandas as pd

# 후처리
# 0~1 -> 1로 바꾸고 + 열 바꾸기

# 1. Read the result file
data = pd.read_csv("./result/submission_patchtst_storeadapt.csv")

# 2. Replace values between 0 and 1 (inclusive) with 1, except first row/col
data.iloc[:, 1:] = data.iloc[:, 1:].applymap(lambda x: 1 if 0 <= x <= 1 else x)

# 3. Read sample_submission and copy its first column to data
sample = pd.read_csv("./result/sample_submission.csv")
data.iloc[:, 0] = sample.iloc[:, 0]
# 3-1. Replace the leftmost column name with that from sample
data.columns.values[0] = sample.columns[0]

# 4. Save to new file
data.to_csv("./result/patchtst_storeadapt_final.csv", index=False, encoding="utf-8-sig")

/tmp/ipykernel_408051/1807579490.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data.iloc[:, 1:] = data.iloc[:, 1:].applymap(lambda x: 1 if 0 <= x <= 1 else x)


: 